In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [8]:
import pandas as pd
import os
import plotly.express as px
# from dash import Input, Output, doc, html
# from dash import Dash
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [9]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 40.2 MB/s eta 0:00:0000:0100:01


In [10]:
from dash import Dash
from dash import Input, Output, dcc, html

**WRANGLE FUNCTION**

In [11]:
def wrangle(filepath):  
    """Read SCF data file into ``DataFrame``.

    Returns only credit fearful households whose net worth is less than $2 million.

    Parameters
    ----------
    filepath : str
        Location of CSV file.
    """
    df = pd.read_csv(filepath)
    # create mask
    mask = (df["TURNFEAR"]== 1) & (df["NETWORTH"] < 2e6)
    # subset Dataframe
    df = df[mask]
    return df

In [13]:
wrangle?

Signature: wrangle(filepath)
Docstring:
Read SCF data file into ``DataFrame``.

Returns only credit fearful households whose net worth is less than $2 million.

Parameters
----------
filepath : str
    Location of CSV file.
File:      /tmp/ipykernel_48/2248270750.py
Type:      function


In [20]:
df = wrangle("/kaggle/input/datasets/paulajibola/kmeans-and-pca-dataset/SCFP2019.csv")
print(df.shape)
df.head()

(4418, 351)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
5,2,21,3790.476607,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
6,2,22,3798.868505,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,3,2,2
7,2,23,3799.468393,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
8,2,24,3788.076005,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
9,2,25,3793.066589,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2


**BUILD DASHBOARD AND WEB APPLICATION**

It includes:

* **The Presentation Layer:** the part that the User sees.
* **The Business (Application) / Logic Layer:** the part that does all the work.
* **The Service Layer:** the part that connects the Presentation Layer to the Service Layer.

**FUNCTION FOR THE BUSINESS LAYER**

**FUNCTION FOR THE PRESENTATION LAYER**

In [21]:
def trimmed_var(dataframe, proportion=0.1):
    """
    Calculate sample variance after dropping a specified proportion

    from both the lower and upper tails of a pandas dataframe
    """
    # drop missing values
    df = df.dropna()

    # find the quantile thresholds
    low_thresh = df.quantile(proportion)
    high_thresh = df.quantile(1 - proportion)

    # keep only data points inside the bounds and calculate variance
    trimmed_data = df[(df >= low_thresh) & (df <= high_thresh)]

    return trimmed_data.var()

**THE SERVICE LAYER**